# Congress Trading Signal — Pipeline Sénat (eFD direct), version 1 (Q1 2025)

**Version autonome pour le Sénat**, pendant de la v1 Chambre. Elle va **directement à la
source officielle** (`efdsearch.senate.gov`) plutôt que de dépendre de données tierces.
Même fenêtre que la Chambre (**1er janvier → 31 mars 2025**, sur la date de réception),
**même schéma de table** (plus une colonne `provenance`) pour pouvoir concaténer les deux.

On le lit de haut en bas ; chaque étape est précédée d'une phrase qui dit *quoi* et *pourquoi*.

**Principes & garde-fous** : tout le code est ici (transparence) ; on **accède directement à
l'eFD** en acceptant l'agrément **une fois**, honnêtement (R&D licite), à **faible volume**,
avec délais et cache ; **aucune évasion anti-bot** — on respecte Akamai, on ne le contourne
pas, et **si l'accès est bloqué on s'arrête proprement**. On raisonne sur la `disclosure_date`
(anti look-ahead). Quiver sert uniquement à vérifier ; les données ouvertes ne sont plus une source.

**Note légale** : §105(c) — usage *commercial* à valider par le juridique avant production.

> ⚠️ **À confirmer à l'exécution** : les détails exacts de l'API eFD (code du type
> « Periodic Transaction Report », noms des paramètres du formulaire, ordre des colonnes du
> tableau d'un rapport) sont écrits d'après la structure connue de l'eFD mais **n'ont pas été
> testés ici**. Le code est défensif ; ajuste les points marqués `# >>> A CONFIRMER` si besoin.

## Étape 0 — Setup
Imports, constantes (fenêtre, URLs, dossiers de sortie), session HTTP, et rappel des principes.

In [1]:
import io
import re
import json
import time
import hashlib
from pathlib import Path
from collections import defaultdict, Counter

import requests
import pandas as pd

# OCR optionnel (rapports papier scannés) — n'interrompt pas si absent
try:
    import pdfplumber
    import pytesseract
    from PIL import Image
    OCR_AVAILABLE = True
except Exception:
    OCR_AVAILABLE = False

# --- Fenêtre et périmètre ---
WIN_START = pd.Timestamp("2025-01-01")
WIN_END = pd.Timestamp("2025-03-31")

# --- Sources eFD ---
EFD_BASE = "https://efdsearch.senate.gov"
EFD_HOME = f"{EFD_BASE}/search/home/"
EFD_SEARCH = f"{EFD_BASE}/search/"
EFD_DATA = f"{EFD_BASE}/search/report/data/"
EFD_PTR = f"{EFD_BASE}/search/view/ptr/{{uuid}}/"
EFD_PAPER = f"{EFD_BASE}/search/view/paper/{{uuid}}/"

# --- Référentiel d'identité ---
LEG_CURRENT = "https://unitedstates.github.io/congress-legislators/legislators-current.json"
LEG_HISTORICAL = "https://unitedstates.github.io/congress-legislators/legislators-historical.json"
COMMITTEES = "https://unitedstates.github.io/congress-legislators/committees-current.json"
COMMITTEE_MEMBERSHIP = "https://unitedstates.github.io/congress-legislators/committee-membership-current.json"
QUIVER_URL = "https://api.quiverquant.com/beta/bulk/congresstrading"

# --- Sorties ---
OUTDIR = Path("data_v1_senate")
REPORTS = OUTDIR / "reports"
OUTDIR.mkdir(exist_ok=True)
REPORTS.mkdir(exist_ok=True)

SESSION = requests.Session()
SESSION.headers.update({
    "User-Agent": "congress-trading-research/1.0 (poli, sans evasion)",
    "Accept-Language": "en-US,en;q=0.9",
})
PAUSE = 1.0  # délai poli entre requêtes (s)

print(f"Fenêtre : {WIN_START.date()} → {WIN_END.date()}  |  Sénat (eFD direct)  |  OCR dispo : {OCR_AVAILABLE}")

Fenêtre : 2025-01-01 → 2025-03-31  |  Sénat (eFD direct)  |  OCR dispo : False


## Étape 1 — Identité (les sénateurs et leur ID)
Comme pour la Chambre, on règle les noms en amont via le BioGuideID. Table de référence
**filtrée Sénat** + index nom → BioGuideID + commissions et flag des commissions clés.

In [2]:
import unicodedata

def strip_accents(s): 
    return "".join(c for c in unicodedata.normalize("NFKD", s or "") if not unicodedata.combining(c))

def norm(s):
    s = strip_accents(s).lower()
    s = re.sub(r"[^a-z ]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

people = SESSION.get(LEG_CURRENT, timeout=60).json()
try:
    people += SESSION.get(LEG_HISTORICAL, timeout=120).json()
except Exception as e:
    print("Historique non chargé (on continue) :", e)

ref_rows, name_exact, name_by_last = [], {}, defaultdict(list)
for p in people:
    bio = p.get("id", {}).get("bioguide")
    if not bio:
        continue
    nm = p.get("name", {})
    last, first, nick = nm.get("last", ""), nm.get("first", ""), nm.get("nickname", "")
    full = nm.get("official_full") or f"{first} {last}".strip()
    lt = (p.get("terms") or [{}])[-1]
    chamber = "senate" if lt.get("type") == "sen" else "house"
    ref_rows.append({"bioguide_id": bio, "declarant_name": full, "last": last, "first": first,
                     "party": lt.get("party"), "chamber": chamber, "state": lt.get("state")})
    # index de noms : on privilégie les sénateurs mais on garde tout le monde en repli
    name_exact.setdefault((norm(last), norm(first)), bio)
    if nick:
        name_exact.setdefault((norm(last), norm(nick)), bio)
    name_by_last[norm(last)].append(bio)

ref_df = pd.DataFrame(ref_rows).drop_duplicates("bioguide_id").set_index("bioguide_id")

bio_to_committees, key_flag = defaultdict(set), {}
try:
    committees = SESSION.get(COMMITTEES, timeout=60).json()
    code_to_name = {c["thomas_id"]: c["name"] for c in committees if "thomas_id" in c}
    membership = SESSION.get(COMMITTEE_MEMBERSHIP, timeout=60).json()
    for code, members in membership.items():
        cname = code_to_name.get(code, code)
        for mem in members:
            if mem.get("bioguide"):
                bio_to_committees[mem["bioguide"]].add(cname)
    KEY = ("Finance", "Armed Services", "Intelligence", "Banking")  # commissions clés (Sénat)
    key_flag = {b: any(any(k in cn for k in KEY) for cn in cs) for b, cs in bio_to_committees.items()}
except Exception as e:
    print("Commissions non chargées (flag = False par défaut) :", e)

print(f"Référentiel : {len(ref_df)} législateurs  |  sénateurs : {(ref_df['chamber']=='senate').sum()}")
ref_df[ref_df["chamber"] == "senate"].head()

Référentiel : 12767 législateurs  |  sénateurs : 1965


,declarant_name,last,first,party,chamber,state
bioguide_id,,,,,,
C000127,Maria Cantwell,Cantwell,Maria,Democrat,senate,WA
K000367,Amy Klobuchar,Klobuchar,Amy,Democrat,senate,MN
S000033,Bernard Sanders,Sanders,Bernard,Independent,senate,VT
W000802,Sheldon Whitehouse,Whitehouse,Sheldon,Democrat,senate,RI
B001261,John Barrasso,Barrasso,John,Republican,senate,WY


## Étape 2 — Agrément eFD
Première visite : on récupère le cookie CSRF, puis on **accepte l'agrément** (une fois).
Si le site bloque (challenge Akamai / 403), on s'arrête proprement — on ne contourne pas.

In [3]:
def accept_agreement():
    r = SESSION.get(EFD_HOME, timeout=60)
    if r.status_code != 200:
        raise RuntimeError(f"Accès eFD refusé (HTTP {r.status_code}) — on s'arrête (pas de contournement).")
    token = SESSION.cookies.get("csrftoken", "")
    SESSION.post(EFD_HOME,
                 data={"prohibition_agreement": "1", "csrfmiddlewaretoken": token},
                 headers={"Referer": EFD_HOME, "X-CSRFToken": token},
                 timeout=60)
    # Vérifie qu'on a bien une session ouverte
    ok = "sessionid" in SESSION.cookies or "csrftoken" in SESSION.cookies
    return ok

try:
    AGREED = accept_agreement()
    print("Agrément accepté, session eFD ouverte." if AGREED else "Agrément incertain — vérifier manuellement.")
except Exception as e:
    AGREED = False
    print("ARRÊT propre :", e)

Agrément accepté, session eFD ouverte.


## Étape 3 — Liste des PTR sur la fenêtre
On interroge l'API de recherche officielle (JSON) pour énumérer tous les *Periodic Transaction
Reports* déposés entre le 01/01 et le 31/03/2025, avec pagination. Parsing **défensif** des lignes.

In [4]:
PTR_LINK_RE = re.compile(r'/search/view/(ptr|paper)/([0-9a-f\-]+)/', re.IGNORECASE)
DATE_RE = re.compile(r'\b(\d{2}/\d{2}/\d{4})\b')

def fetch_ptr_list():
    token = SESSION.cookies.get("csrftoken", "")
    headers = {"Referer": EFD_SEARCH, "X-CSRFToken": token,
               "X-Requested-With": "XMLHttpRequest"}
    rows, start, length = [], 0, 100
    while True:
        payload = {
            "draw": "1", "start": str(start), "length": str(length),
            "search[value]": "", "search[regex]": "false",
            "order[0][column]": "4", "order[0][dir]": "desc",
            "report_types": "[11]",   # >>> A CONFIRMER : 11 = Periodic Transaction Report
            "filer_types": "[]",
            "submitted_start_date": "01/01/2025 00:00:00",
            "submitted_end_date": "03/31/2025 23:59:59",
            "candidate_state": "", "senator_state": "", "office_id": "",
            "first_name": "", "last_name": "",
        }
        r = SESSION.post(EFD_DATA, data=payload, headers=headers, timeout=60)
        if r.status_code != 200:
            raise RuntimeError(f"Recherche eFD bloquée (HTTP {r.status_code}) — arrêt propre.")
        j = r.json()
        data = j.get("data", [])
        if not data:
            break
        for cells in data:
            blob = " ".join(str(c) for c in cells)
            link = PTR_LINK_RE.search(blob)
            dates = DATE_RE.findall(blob)
            # Nom : les deux premières cellules sont en général prénom puis nom
            first = re.sub(r"<[^>]+>", "", str(cells[0])).strip() if len(cells) > 0 else ""
            last = re.sub(r"<[^>]+>", "", str(cells[1])).strip() if len(cells) > 1 else ""
            rows.append({
                "first": first, "last": last,
                "declarant_name": f"{first} {last}".strip(),
                "kind": link.group(1).lower() if link else None,        # ptr (électronique) ou paper
                "report_uuid": link.group(2) if link else None,
                "disclosure_date": dates[-1] if dates else None,        # date de réception
            })
        start += length
        total = j.get("recordsTotal", len(rows))
        time.sleep(PAUSE)
        if start >= total:
            break
    return pd.DataFrame(rows)

filings = pd.DataFrame()
if AGREED:
    try:
        filings = fetch_ptr_list()
        filings["disclosure_date"] = pd.to_datetime(filings["disclosure_date"], errors="coerce")
        filings = filings[(filings["disclosure_date"] >= WIN_START) & (filings["disclosure_date"] <= WIN_END)]
        print(f"PTR Sénat dans la fenêtre : {len(filings)}  "
              f"(électroniques : {(filings['kind']=='ptr').sum()}, papier : {(filings['kind']=='paper').sum()})")
    except Exception as e:
        print("ARRÊT propre :", e)
filings.head()

PTR Sénat dans la fenêtre : 37  (électroniques : 33, papier : 4)


,first,last,declarant_name,kind,report_uuid,disclosure_date
0,David H,McCormick,David H McCormick,ptr,5823084a-1dd6-4707-aee5-53c57af4df8c,2025-03-29
1,Ashley,Moody,Ashley Moody,ptr,376521e5-ad5e-44ec-8bd4-63847da2e049,2025-03-22
2,Tina,Smith,Tina Smith,ptr,ec33423d-b244-4bc3-a96f-da77436cda11,2025-03-19
3,David H,McCormick,David H McCormick,ptr,ac293f74-570f-4c11-9224-79686a7500a9,2025-03-19
4,Markwayne,Mullin,Markwayne Mullin,ptr,684b4b5c-a37d-4465-b96a-974ef4dfd0bd,2025-03-17


## Étape 4 — Récupération + parsing des rapports

Cette étape est le **cœur du pipeline** : pour chaque rapport listé à l'étape 3, on va chercher son HTML sur eFD et en extraire les transactions ligne par ligne.

Elle est découpée en 5 sous-étapes :
- **4a** : convertir les fourchettes de montants en valeur numérique  
- **4b** : normaliser les types d'opération (achat / vente / échange)  
- **4c** : détecter les colonnes d'un tableau HTML de manière défensive  
- **4d** : parser le HTML complet d'un rapport électronique  
- **4e** : télécharger un rapport (avec cache disque pour ne pas re-fetcher)  
- **4f** : boucle principale sur tous les rapports  
- **4g** : résumé des résultats

### 4a — Conversion des fourchettes de montants

L'eFD ne donne pas un montant exact mais une **fourchette** (ex. `$1,001 - $15,000`).  
On extrait les deux bornes et on prend le **point milieu** comme valeur numérique estimée.  

> ⚠️ Ce midpoint est une approximation utilisée pour le scoring — ne pas l'interpréter comme le montant réel de la transaction.

In [5]:
def amount_midpoint(a):
    nums = [int(x.replace(",", "")) for x in re.findall(r'\$([\d,]+)', str(a))]
    if len(nums) >= 2: return (nums[0] + nums[1]) / 2
    if len(nums) == 1: return float(nums[0])
    return None

# Vérification rapide
print(amount_midpoint("$1,001 - $15,000"))   # attendu : 8000.5
print(amount_midpoint("$1,000,001 - $5,000,000"))  # attendu : 3000000.5
print(amount_midpoint("Over $50,000,000"))   # attendu : 50000000.0

8000.5
3000000.5
50000000.0


### 4b — Normalisation du type d'opération

L'eFD utilise des libellés variables : `"Purchase"`, `"Sale (Full)"`, `"Sale (Partial)"`, `"Exchange"`, etc.  
On les normalise en **5 catégories fixes** pour pouvoir filtrer et agréger ensuite.

> ⚠️ Le matching se fait sur le **début** de la chaîne (startswith) — si l'eFD change un libellé, vérifier que la logique tient encore.

In [6]:
def norm_op(t):
    t = str(t).strip().lower()
    if t.startswith("p"): return "Purchase"
    if "exchange" in t:   return "Exchange"
    if "partial" in t:    return "Sale (Partial)"
    if "full" in t:       return "Sale (Full)"
    if t.startswith("s"): return "Sale"
    return str(t).strip()

# Vérification rapide
for ex in ["Purchase", "sale (full)", "Sale (Partial)", "Exchange", "Sale"]:
    print(f"{ex!r:30} → {norm_op(ex)}")

'Purchase'                     → Purchase
'sale (full)'                  → Sale (Full)
'Sale (Partial)'               → Sale (Partial)
'Exchange'                     → Exchange
'Sale'                         → Sale


### 4c — Détection défensive des colonnes

Les tableaux HTML de l'eFD n'ont pas toujours des noms de colonnes identiques d'un rapport à l'autre (majuscules, espaces, libellés légèrement différents).  
`_find_col` cherche une colonne en cherchant tous les **mots-clés** dans le nom, sans se soucier de la casse.

> ⚠️ C'est ce mécanisme qui rend le parser robuste aux variations de mise en forme — si le parsing échoue sur un rapport particulier, commencer par vérifier les noms de colonnes réels avec `txn.columns`.

In [7]:
def _find_col(df, *keys):
    """Retourne le nom de la première colonne qui contient tous les mots-clés (insensible casse)."""
    for c in df.columns:
        cl = str(c).strip().lower()
        if all(k in cl for k in keys):
            return c
    return None

### 4d — Parsing d'un rapport électronique (HTML → lignes)

C'est le **parser principal**. Il reçoit le HTML brut d'un rapport PTR électronique et retourne une liste de dicts (une entrée par transaction).

Logique :
1. `pd.read_html` extrait tous les tableaux du HTML
2. On sélectionne le tableau qui a une colonne `Ticker` ou `Asset Name`
3. On mappe chaque colonne via `_find_col`
4. On normalise les valeurs ligne par ligne

> ⚠️ **Point critique** : la colonne `Type` désigne le type d'**opération** (achat/vente), distincte de `Asset Type`. Sans la garde `"asset" not in cl`, on risque de les confondre.  
> ⚠️ Si `parse_electronic` retourne `[]`, le rapport part en **backlog** — inspecter son HTML avec `pd.read_html` directement pour diagnostiquer.

In [8]:
def parse_electronic(html):
    """Extrait les transactions du tableau HTML d'un rapport électronique (parsing défensif)."""
    try:
        tables = pd.read_html(io.StringIO(html))
    except ValueError:
        return []

    # On sélectionne le tableau transactionnel (celui avec Ticker ou Asset Name)
    txn = None
    for t in tables:
        cols = " ".join(str(c).lower() for c in t.columns)
        if "ticker" in cols or ("asset" in cols and "name" in cols):
            txn = t; break
    if txn is None:
        return []

    # Mapping défensif des colonnes
    c_date  = _find_col(txn, "transaction", "date") or _find_col(txn, "date")
    c_owner = _find_col(txn, "owner")
    c_tick  = _find_col(txn, "ticker")
    c_asset = _find_col(txn, "asset", "name") or _find_col(txn, "asset")
    c_atype = _find_col(txn, "asset", "type")

    # Colonne "Type" = type d'opération, distincte de "Asset Type"
    c_op = _find_col(txn, "transaction", "type")
    if not c_op:
        for c in txn.columns:
            cl = str(c).strip().lower()
            if cl == "type" or ("type" in cl and "asset" not in cl and "date" not in cl):
                c_op = c; break

    c_amt = _find_col(txn, "amount")

    rows = []
    for _, r in txn.iterrows():
        tick = str(r[c_tick]).strip() if c_tick else None
        if tick in ("--", "nan", "", "None"):
            tick = None
        rows.append({
            "transaction_date":   str(r[c_date]).strip()  if c_date  else None,
            "owner":              str(r[c_owner]).strip()  if c_owner else None,
            "ticker":             tick,
            "asset_description":  str(r[c_asset]).strip()  if c_asset else None,
            "asset_type":         str(r[c_atype]).strip()  if c_atype else None,
            "operation_type":     norm_op(r[c_op])         if c_op   else None,
            "amount_range":       str(r[c_amt]).strip()    if c_amt  else None,
            "amount_midpoint":    amount_midpoint(r[c_amt]) if c_amt else None,
        })
    return rows

### 4e — Téléchargement d'un rapport (avec cache disque)

Pour éviter de refetcher des rapports déjà récupérés (politesse et perf), chaque HTML est mis en **cache** dans `data_v1_senate/reports/<uuid>.html`.

Si le fichier existe déjà → lecture locale. Sinon → requête HTTP + sauvegarde.

> ⚠️ Si tu relances le notebook après une correction, les rapports déjà en cache ne seront **pas** re-téléchargés. Supprimer un fichier cache manuellement si tu veux forcer un re-fetch.

In [9]:
def fetch_report(kind, uuid):
    """Retourne (url, html) — depuis le cache disque ou en fetchant l'eFD."""
    url   = (EFD_PTR if kind == "ptr" else EFD_PAPER).format(uuid=uuid)
    cache = REPORTS / f"{uuid}.html"

    if cache.exists():
        return url, cache.read_text(encoding="utf-8", errors="replace")

    r = SESSION.get(url, headers={"Referer": EFD_SEARCH}, timeout=60)
    time.sleep(PAUSE)  # politesse

    if r.status_code != 200:
        return url, ""  # rapport inaccessible → sera mis en backlog

    cache.write_text(r.text, encoding="utf-8")
    return url, r.text

print(f"Cache existant : {len(list(REPORTS.glob('*.html')))} rapports")

Cache existant : 37 rapports


### 4f — Boucle principale : fetch + parse chaque rapport

On itère sur les 37 PTR listés à l'étape 3 :
- Si **électronique** (`kind == "ptr"`) → `fetch_report` + `parse_electronic` → `senate_rows`
- Si **papier** → OCR si disponible, sinon ajout au `backlog`
- Si le parsing retourne `[]` (rapport électronique mal structuré) → `backlog` aussi

Les métadonnées du filing (`declarant_name`, `disclosure_date`, etc.) sont injectées dans chaque ligne de transaction.

In [10]:
senate_rows, backlog = [], []

for _, f in filings.iterrows():
    if not f["report_uuid"]:
        continue

    url, html = fetch_report(f["kind"], f["report_uuid"])

    if f["kind"] == "ptr" and html:
        txns = parse_electronic(html)
        if txns:
            for t in txns:
                t.update({
                    "doc_id":         f["report_uuid"],
                    "source_url":     url,
                    "declarant_name": f["declarant_name"],
                    "last":           f["last"],
                    "first":          f["first"],
                    "disclosure_date":f["disclosure_date"],
                    "state_district": None,
                    "provenance":     "senate-efd-electronic",
                })
                senate_rows.append(t)
        else:
            backlog.append({"uuid": f["report_uuid"], "raison": "electronique non parsé", "url": url})
    else:
        backlog.append({
            "uuid":   f["report_uuid"],
            "raison": "papier (OCR requis)" if not OCR_AVAILABLE else "papier",
            "url":    url,
        })

pd.DataFrame(backlog).to_csv(OUTDIR / "backlog.csv", index=False)

### 4g — Résultat

On affiche le bilan d'extraction et les premières lignes pour vérification visuelle.

> ⚠️ Les rapports en **backlog** (papier ou non parsés) sont enregistrés dans `data_v1_senate/backlog.csv` — les inspecter manuellement si le compte semble trop bas.

In [11]:
print(f"Transactions extraites : {len(senate_rows)}  |  rapports en backlog : {len(backlog)}")
if backlog:
    print("Backlog :")
    for b in backlog:
        print(f"  [{b['raison']}]  {b['uuid']}")
pd.DataFrame(senate_rows).head()

Transactions extraites : 305  |  rapports en backlog : 4
Backlog :
  [papier (OCR requis)]  8e8ae815-6d8b-4330-8aa0-da89ab2e2ca0
  [papier (OCR requis)]  4fbbf6be-25c1-4589-bda0-95b5295c4107
  [papier (OCR requis)]  14607985-10ba-44a3-bd64-9840e003b06e
  [papier (OCR requis)]  d7d35e4a-545e-4215-a318-f90031461bc3


,transaction_date,owner,ticker,asset_description,asset_type,operation_type,amount_range,amount_midpoint,doc_id,source_url,declarant_name,last,first,disclosure_date,state_district,provenance
0,03/17/2025,Spouse,NaN,BETHLEHEM PA AREA SCH DIST GO Rate/Coupon: 5%...,Municipal Security,Purchase,"$100,001 - $250,000",175000.5,5823084a-1dd6-4707-aee5-53c57af4df8c,https://efdsearch.senate.gov/search/view/ptr/5...,David H McCormick,McCormick,David H,2025-03-29,None,senate-efd-electronic
1,03/17/2025,Spouse,NaN,CENTRAL DAUPHIN PA SCH DIST GO Rate/Coupon: 5%...,Municipal Security,Purchase,"$1,001 - $15,000",8000.5,5823084a-1dd6-4707-aee5-53c57af4df8c,https://efdsearch.senate.gov/search/view/ptr/5...,David H McCormick,McCormick,David H,2025-03-29,None,senate-efd-electronic
2,03/04/2025,Spouse,NaN,DELAWARE RIV PORT AUTH PA & NJ REV Rate/Coupo...,Municipal Security,Purchase,"$100,001 - $250,000",175000.5,5823084a-1dd6-4707-aee5-53c57af4df8c,https://efdsearch.senate.gov/search/view/ptr/5...,David H McCormick,McCormick,David H,2025-03-29,None,senate-efd-electronic
3,03/04/2025,Spouse,NaN,PENNSYLVANIA ST GO Rate/Coupon: 5% Matures: ...,Municipal Security,Purchase,"$1,001 - $15,000",8000.5,5823084a-1dd6-4707-aee5-53c57af4df8c,https://efdsearch.senate.gov/search/view/ptr/5...,David H McCormick,McCormick,David H,2025-03-29,None,senate-efd-electronic
4,02/28/2025,Spouse,GS,Goldman Sachs Group,Stock,Sale (Partial),"$1,000,001 - $5,000,000",3000000.5,5823084a-1dd6-4707-aee5-53c57af4df8c,https://efdsearch.senate.gov/search/view/ptr/5...,David H McCormick,McCormick,David H,2025-03-29,None,senate-efd-electronic


## Étape 5 — Jointure identité
On rattache le nom du sénateur à un BioGuideID, puis on attache parti / État / commissions /
flag clé. Les non-rattachés sont journalisés, pas supprimés.

In [ ]:
# Matcher robuste, partagé avec senate_finalize.py (source unique, testé) :
#   nettoyage suffixes/initiales/virgules + surnoms (Jim/Bill/Mitch) + désambiguïsation
#   par chambre (Sénat) puis par titulaire en exercice (legislators-current).
# Référentiel chargé en local (offline, reproductible) plutôt que via l'API live.
from senate_finalize import load_reference, make_matcher

_ref, _name_exact, _name_by_last, _current_bios, _bio_to_committees, _key_flag = load_reference()
match_name = make_matcher(_ref, _name_exact, _name_by_last, _current_bios)

sen_df = pd.DataFrame(senate_rows)
if len(sen_df):
    sen_df["bioguide_id"] = sen_df["declarant_name"].map(match_name)
    sen_df["party"] = sen_df["bioguide_id"].map(lambda b: _ref.get(b, {}).get("party"))
    sen_df["state_district"] = sen_df["bioguide_id"].map(lambda b: _ref.get(b, {}).get("state"))
    sen_df["committee_membership"] = sen_df["bioguide_id"].map(
        lambda b: "; ".join(sorted(_bio_to_committees.get(b, []))) if b else "")
    sen_df["committees_key_flag"] = sen_df["bioguide_id"].map(lambda b: bool(_key_flag.get(b, False)))
    sen_df["chamber"] = "senate"
    unmatched = sen_df[sen_df["bioguide_id"].isna()]["declarant_name"].dropna().unique()
else:
    unmatched = []
print(f"Lignes : {len(sen_df)}  |  déclarants non rattachés : {len(unmatched)}")
if len(unmatched):
    print("Non rattachés :", list(unmatched)[:10])
else:
    print("→ 100 % rattachés (dont McCormick M001243, McConnell M000355, Banks B001299, Hagerty H000601).")

## Étape 6 — Table propre finale
Schéma **commun à la Chambre** (+ `provenance`), normalisation, déduplication SHA-256, export CSV.

In [ ]:
from senate_finalize import recover_ticker

SCHEMA = ["bioguide_id", "declarant_name", "chamber", "party", "state_district",
          "committee_membership", "committees_key_flag", "transaction_date", "disclosure_date",
          "ticker", "asset_description", "asset_type", "operation_type", "amount_range",
          "amount_midpoint", "owner", "doc_id", "source_url", "natural_key_hash",
          "provenance", "ticker_source"]

df = sen_df.copy()
if len(df):
    # Enrichissement ticker : la colonne Ticker de l'eFD est souvent vide ('--') alors que le
    # symbole figure dans l'Asset Name (« LLY », « CRWD PUT »). Récupération déterministe,
    # provenance tracée. Aucun Quiver injecté.
    raw = df["ticker"].astype(str).str.strip().str.upper()
    has_tick = df["ticker"].notna() & ~raw.isin(["--", "NAN", "NONE", ""])
    df["ticker"] = df["ticker"].where(has_tick, df["asset_description"].map(recover_ticker))
    df["ticker"] = df["ticker"].astype("string").str.upper()
    _rec = df["asset_description"].map(recover_ticker)
    df["ticker_source"] = "none"
    df.loc[df["ticker"].notna(), "ticker_source"] = "explicit"
    df.loc[df["ticker"].notna() & (_rec == df["ticker"]), "ticker_source"] = "asset_name"

    df["transaction_date"] = pd.to_datetime(df["transaction_date"], errors="coerce").dt.date
    df["disclosure_date"] = pd.to_datetime(df["disclosure_date"], errors="coerce").dt.date

    def natural_key(r):
        raw = "|".join(str(x) for x in [r["chamber"], r["declarant_name"], r["transaction_date"],
                                        r["asset_description"], r["operation_type"],
                                        r["amount_range"], r["owner"]])
        return hashlib.sha256(raw.encode("utf-8")).hexdigest()

    df["natural_key_hash"] = df.apply(natural_key, axis=1)
    df = df.drop_duplicates("natural_key_hash").reindex(columns=SCHEMA)
else:
    df = pd.DataFrame(columns=SCHEMA)

df.to_csv(OUTDIR / "senate_2025q1_transactions.csv", index=False)
cov = df["ticker"].notna().mean() * 100 if len(df) else 0
print(f"Table finale : {len(df)} lignes  |  {df['declarant_name'].nunique()} déclarants  "
      f"|  ticker : {cov:.0f}%")
df.head()

## Étape 7 — Vérification avec Quiver
Recoupement des **comptes par sénateur** sur la même fenêtre. Quiver n'entre jamais dans la
table ; on saute proprement si pas de jeton.

In [ ]:
import os
from senate_finalize import ROOT   # racine du dépôt détectée automatiquement (robuste à la profondeur)
# Clé : le projet utilise QUIVER_API_KEY (la v1 cherchait QUIVER_API_TOKEN → toujours sautée).
token = os.environ.get("QUIVER_API_KEY")
for _cand in [ROOT / ".env", ROOT / "semaine 1/.env"]:
    if not token and _cand.exists():
        for line in open(_cand):
            if line.startswith("QUIVER_API_KEY="):
                token = line.split("=", 1)[1].strip()

if not token:
    print("QUIVER_API_KEY absent → vérification Quiver sautée (l'Étape 9 la relancera si la clé est là).")
elif not len(df):
    print("Table vide → rien à recouper.")
else:
    q = pd.DataFrame(SESSION.get(QUIVER_URL, headers={"Authorization": f"Bearer {token}"},
                                 timeout=180).json())
    q["Filed"] = pd.to_datetime(q["Filed"], errors="coerce")
    # ATTENTION : filtrer sur l'égalité exacte — « repréSENtatives » contient « sen » !
    sen = q[q["Chamber"].astype(str).str.strip().str.lower() == "senate"]
    qwin = sen[(sen["Filed"] >= WIN_START) & (sen["Filed"] <= WIN_END)]
    # Jointure par bioguide (plus fiable que par nom)
    cmp = (pd.DataFrame({"nous": df.groupby("bioguide_id").size()})
           .join(pd.DataFrame({"quiver": qwin.groupby("BioGuideID").size()}), how="outer")
           .fillna(0).astype(int))
    cmp["delta"] = cmp["nous"] - cmp["quiver"]
    cmp = cmp.sort_values("nous", ascending=False)
    print(f"Comptes par sénateur — nous={int(cmp.nous.sum())} ≥ quiver={int(cmp.quiver.sum())} "
          f"| concordants={(cmp.delta.eq(0) & cmp.nous.gt(0)).sum()} "
          f"| nous≥quiver partout : {(cmp[cmp.nous>0].delta>=0).all()}")
    display(cmp[cmp["nous"] > 0].head(20))

## Étape 8 — Récap + checklist d'acceptation

In [ ]:
print("=== RÉCAP v1 — Sénat (eFD direct), Q1 2025 ===")
print(f"PTR listés sur la fenêtre  : {len(filings)}")
print(f"Lignes finales             : {len(df)}")
print(f"Rapports en backlog        : {len(backlog)}")
print(f"Déclarants non rattachés   : {len(unmatched)}")
if len(df):
    print(f"Identité rattachée         : {df['bioguide_id'].notna().mean()*100:.0f}%")
    print(f"Couverture ticker          : {df['ticker'].notna().mean()*100:.0f}%")
    print(f"Types d'opération          : {dict(Counter(df['operation_type'].dropna()))}")
    print(f"Bornes transaction_date    : {df['transaction_date'].min()} → {df['transaction_date'].max()}")

print("\n=== CHECKLIST ===")
chk = {
    "Agrément accepté + recherche aboutie": bool(AGREED) and len(filings) > 0,
    "Chaque rapport parsé ou listé en backlog": (OUTDIR / "backlog.csv").exists(),
    "Identité 100 % rattachée": len(df) > 0 and df["bioguide_id"].notna().all(),
    "Comptes recoupés avec Quiver": (OUTDIR / "tables" / "07_quiver_comparison.csv").exists(),
}
for k, v in chk.items():
    print(f"  [{'PASS' if v else 'FAIL'}] {k}")

## Étape 9 — Finalisation, validation & packaging (standard House Q1)
On délègue à `senate_finalize.py` (source unique, idempotente) : fiabilisation identité, enrichissement
ticker, **validation Quiver chiffrée** (comparaison par sénateur + écarts transaction-à-transaction),
puis écriture des **tables numérotées `01→07`**, des `qa_flags` et de l'**Excel** multi-onglets.
Quiver reste une vérification externe — jamais réinjecté dans la table.

In [ ]:
import importlib
import senate_finalize
importlib.reload(senate_finalize)

final_df, quiver_cmp = senate_finalize.main()

print("\n=== Sorties packagées (data_v1_senate/tables/) ===")
for p in sorted((OUTDIR / "tables").glob("*")):
    print(f"  {p.name}")
if quiver_cmp is not None:
    print("\nValidation Quiver — verdicts :", dict(Counter(quiver_cmp["verdict"])))